In [3]:
using Ket
using Hypatia



In [4]:
# Fast computation of f(m, N, k_list)
function f_fast(m::Int, N::Int, k_list::Vector{Int})
    total = 0.0
    for s in 0:(N - m - 1)
        sign = 1
        for k in k_list
            if s >= k - 1
                exponent = binomial(s, k - 1)
                sign *= (-1)^exponent
            end
        end
        total += binomial(N - m - 1, s) * sign
    end
    return total / 2.0^(N - m - 1)
end


function C(n::Int, N::Int, k_list::Vector{Int})
    s = 0.0
    for r in 0:n
        s += (-1)^(n - r) * binomial(n, r) * f_fast(r, N, k_list)
    end
    return s
end

C (generic function with 1 method)

# Determine the coefficients in Correlator notation

In [5]:
using Combinatorics


function bell_functional(N::Int, k_list::Vector{Int})
    β = Dict{NTuple{N, Int}, Float64}()
    V = 1:N

    for m in 1:N-1
        coeff = C(m, N, k_list) / 2 
        
        for v in combinations(V, m+1)
            for k in v
                # ---- Term 1 ----
                # (|v|-1) * (A_0^{(k)} + A_1^{(k)}) * \prod_{i \ne k} A_3^{(i)}
                for obs in 1:2
                    # Allocation-free tuple generation
                    tup = ntuple(N) do idx
                        if idx == k
                            return obs
                        elseif idx in v
                            return 4 # A_3
                        else
                            return 0 # Identity
                        end
                    end
                    
                    β[tup] = get(β, tup, 0.0) + coeff 
                end

                # ---- Term 2 ----
                # \sum_{i \ne k} (A_0^{(k)} - A_1^{(k)}) * A_2^{(i)} * \prod_{j \ne i, k} A_3^{(j)}
                for i in v
                    if i == k
                        continue
                    end
                    for (obs, sign) in zip((1, 2), (1.0, -1.0))
                        # Allocation-free tuple generation
                        tup = ntuple(N) do idx
                            if idx == k
                                return obs
                            elseif idx == i
                                return 3 # A_2
                            elseif idx in v
                                return 4 # A_3
                            else
                                return 0 # Identity
                            end
                        end
                        
                        β[tup] = get(β, tup, 0.0) + coeff * sign/m
                    end
                end
            end
        end
    end

    return β
end

function dict_to_tensor(β::Dict{NTuple{M,Int},Float64}, N::Int) where M
    @assert M == N
    # Explicitly creating a D^N array where D=5
    T = zeros(Float64, ntuple(_ -> 5, N)...)

    for (idx, val) in β
        shifted = ntuple(i -> idx[i] + 1, N)
        T[shifted...] = val
    end

    return T
end

dict_to_tensor (generic function with 1 method)

In [11]:
N = 4
k_list = [3]

bell_dict = bell_functional(N, k_list)
FC = dict_to_tensor(bell_dict, N)
println("Functional Value for Hypergraph strategy: ", sqrt(2)*N)  
println("Classical Bound: ", local_bound(FC; correlation = true, marg = true))
println("Quantum Bound: ", tsirelson_bound(FC, 2; solver = Hypatia.Optimizer))

Functional Value for Hypergraph strategy: 5.656854249492381
Classical Bound: 6.666666666666667
Quantum Bound: (7.061313461800763, [1.0 0.0 0.0 0.0 0.0; 0.0 0.5704016757717131 -0.7157619754682182 0.7505191220197839 0.14097136680464217; 0.0 -0.7157619755081257 0.5704016756931295 -0.7505191220102947 0.1409713668859396; 0.0 0.7505191220954066 -0.750519122025936 0.8759055614793757 -4.775715922755218e-11; 0.0 0.14097136671126223 0.14097136693197065 -1.1610604454665741e-10 -0.2734298498776193;;; 0.0 0.5704016759075923 -0.7157619755590447 0.7505191221520741 0.14097136676094954; 0.5704016758192804 0.0 0.0 0.0 0.0; -0.7157619755377883 0.0 0.0 0.0 0.0; 0.750519122140473 0.0 0.0 0.0 0.0; 0.14097136669389831 0.0 0.0 0.0 0.0;;; 0.0 -0.7157619755475456 0.5704016757309411 -0.7505191220553616 0.1409713668874993; -0.7157619754863815 0.0 0.0 0.0 0.0; 0.5704016757187598 0.0 0.0 0.0 0.0; -0.7505191220514912 0.0 0.0 0.0 0.0; 0.14097136692472909 0.0 0.0 0.0 0.0;;; 0.0 0.7505191221068007 -0.7505191220349329 0